0 — Paths + PYTHONPATH setup

In [1]:
import os, sys
from pathlib import Path

THESIS_ROOT = Path("/home/psivakumar/thesis").resolve()
R2GEN_DIR   = (THESIS_ROOT / "code" / "R2GenCMN").resolve()

assert THESIS_ROOT.exists(), f"THESIS_ROOT not found: {THESIS_ROOT}"
assert R2GEN_DIR.exists(), f"R2GenCMN repo not found: {R2GEN_DIR}"

os.chdir(R2GEN_DIR)

if str(R2GEN_DIR) not in sys.path:
    sys.path.insert(0, str(R2GEN_DIR))

print("CWD:", Path.cwd())
print("R2GenCMN added to sys.path ✅")
print("modules exists:", (R2GEN_DIR/"modules").exists())
print("models  exists:", (R2GEN_DIR/"models").exists())

CWD: /home/psivakumar/thesis/code/R2GenCMN
R2GenCMN added to sys.path ✅
modules exists: True
models  exists: True


1 — Imports + device

In [2]:
import json
import torch
from torch.utils.data import DataLoader

from modules.tokenizers import Tokenizer
from modules.datasets import IuxrayMultiImageDataset
from models.models import BaseCMNModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


2 — Paths for annotation + images + checkpoint

In [3]:
# Annotation in dict format (train/val/test keys)
ANN_PATH = THESIS_ROOT / "data" / "iu_xray" / "annotation_r2gencmn_2view_r2gencmnformat.json"
assert ANN_PATH.exists(), f"Missing annotation: {ANN_PATH}"

# Image root used by dataset (contains "images/..." symlink)
IMAGE_DIR = R2GEN_DIR / "data" / "iu_xray"
assert IMAGE_DIR.exists(), f"Missing IMAGE_DIR: {IMAGE_DIR}"
assert (IMAGE_DIR / "images").exists(), f"Missing images link: {IMAGE_DIR/'images'}"

# Trained checkpoint (from your 5-epoch run)
CKPT_PATH = R2GEN_DIR / "outputs" / "iu_xray_r2gencmn_baseline_5ep" / "current_checkpoint.pth"
assert CKPT_PATH.exists(), f"Missing checkpoint: {CKPT_PATH}"

print("ANN_PATH :", ANN_PATH)
print("IMAGE_DIR:", IMAGE_DIR)
print("CKPT_PATH:", CKPT_PATH)

ANN_PATH : /home/psivakumar/thesis/data/iu_xray/annotation_r2gencmn_2view_r2gencmnformat.json
IMAGE_DIR: /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray
CKPT_PATH: /home/psivakumar/thesis/code/R2GenCMN/outputs/iu_xray_r2gencmn_baseline_5ep/current_checkpoint.pth


3 — Args object (matches what repo expects)

In [4]:
import torch
from pathlib import Path

CKPT_PATH = Path("/home/psivakumar/thesis/code/R2GenCMN/outputs/iu_xray_r2gencmn_baseline_5ep/current_checkpoint.pth")
ckpt = torch.load(CKPT_PATH, map_location="cpu")

print("Checkpoint keys:", list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))

# Print possible config/args fields if they exist
for k in ["args", "config", "cfg", "hparams", "opt"]:
    if isinstance(ckpt, dict) and k in ckpt:
        v = ckpt[k]
        print(f"\nFound '{k}' in checkpoint. Type:", type(v))
        try:
            if hasattr(v, "__dict__"):
                print("args.__dict__ keys (sample):", list(v.__dict__.keys())[:50])
            elif isinstance(v, dict):
                print("dict keys (sample):", list(v.keys())[:50])
        except Exception as e:
            print("Could not inspect:", e)

/tmp/ipykernel_525487/727354247.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT_PATH, map_location="cpu")


Checkpoint keys: ['epoch', 'state_dict', 'optimizer', 'monitor_best']


In [5]:
class Args:
    # ----------------------------
    # Required paths / dataset
    # ----------------------------
    dataset_name = "iu_xray"
    ann_path = str(ANN_PATH)
    image_dir = str(IMAGE_DIR)
    use_bn = False
    # ----------------------------
    # Tokenizer / sequence
    # ----------------------------
    threshold = 3
    max_seq_length = 100
    pad_idx = 0
    bos_idx = 0
    eos_idx = 0
    unk_idx = 27

    # ----------------------------
    # DataLoader
    # ----------------------------
    num_workers = 2
    topk = 32

    # ----------------------------
    # Visual extractor
    # ----------------------------
    visual_extractor = "resnet101"
    visual_extractor_pretrained = True
    d_vf = 2048  # resnet101 feature dim

    # ----------------------------
    # Encoder/Decoder / Attention model params
    # (these names are referenced in att_model.py / base_cmn.py)
    # ----------------------------
    d_model = 512
    d_ff = 512           # in this repo: rnn_size = d_ff
    num_layers = 3
    num_heads = 8

    # Dropouts
    drop_prob_lm = 0.1   # REQUIRED by att_model.py
    dropout = 0.1

    # ----------------------------
    # Decoding / generation params
    # ----------------------------
    beam_size = 3
    sample_method = "greedy"
    temperature = 1.0

    # ----------------------------
    # Trainer placeholders (not used in inference but referenced)
    # ----------------------------
    monitor_metric = "BLEU_4"
    monitor_mode = "max"
    early_stop = 0
    save_period = 1
    bos_idx = False

args = Args()
print("Args ready ✅")
print("visual_extractor:", args.visual_extractor, "| pretrained:", args.visual_extractor_pretrained)
print("d_vf:", args.d_vf, "| d_model:", args.d_model, "| d_ff:", args.d_ff)
print("drop_prob_lm:", args.drop_prob_lm, "| num_layers:", args.num_layers)

Args ready ✅
visual_extractor: resnet101 | pretrained: True
d_vf: 2048 | d_model: 512 | d_ff: 512
drop_prob_lm: 0.1 | num_layers: 3


In [ ]:
# Special token indices for this repo/tokenizer
args.pad_idx = 0
args.bos_idx = 0
args.eos_idx = 0
args.unk_idx = tokenizer.token2idx.get("<unk>", 27)

print("✅ Special indices set:")
print("pad_idx:", args.pad_idx)
print("bos_idx:", args.bos_idx)
print("eos_idx:", args.eos_idx)
print("unk_idx:", args.unk_idx)

4 — Load annotation dict (sanity)

In [9]:
with open(args.ann_path) as f:
    ann = json.load(f)

print("Keys:", ann.keys())
print("Sizes:", {k: len(v) for k, v in ann.items()})

Keys: dict_keys(['train', 'val', 'test'])
Sizes: {'train': 2131, 'val': 53, 'test': 44}


5 — Build tokenizer

In [10]:
tokenizer = Tokenizer(args)

# Some repos don't implement get_vocab_size; so we handle both cases safely
vocab_size = getattr(tokenizer, "get_vocab_size", None)
if callable(vocab_size):
    print("Vocab size:", tokenizer.get_vocab_size())
else:
    # fallback: try reading internal dict
    if hasattr(tokenizer, "token2idx"):
        print("Vocab size:", len(tokenizer.token2idx))
    else:
        print("Tokenizer built ✅ (vocab size method not available)")

Vocab size: 928


6 — Build TEST dataset + dataloader

In [11]:
import torch

def r2gen_collate_fn(batch):
    """
    Custom collate function for IuxrayMultiImageDataset
    """
    image_ids, images, report_ids, report_masks, seq_lengths = zip(*batch)

    # images: list of [2, 3, 224, 224] → [B, 2, 3, 224, 224]
    images = torch.stack(images, dim=0)

    # Pad reports
    max_len = max(seq_lengths)

    padded_ids = torch.zeros(len(report_ids), max_len, dtype=torch.long)
    padded_masks = torch.zeros(len(report_masks), max_len, dtype=torch.long)

    for i, (ids, masks) in enumerate(zip(report_ids, report_masks)):
        length = len(ids)
        padded_ids[i, :length] = torch.tensor(ids, dtype=torch.long)
        padded_masks[i, :length] = torch.tensor(masks, dtype=torch.long)

    return image_ids, images, padded_ids, padded_masks, torch.tensor(seq_lengths)

In [12]:
from torchvision import transforms

# Standard ImageNet normalization (because ResNet101 pretrained)
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print("Transform ready ✅")

Transform ready ✅


In [13]:
test_dataset = IuxrayMultiImageDataset(
    args=args,
    tokenizer=tokenizer,
    split="test",
    transform=img_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=args.num_workers
)

print("Test dataset size:", len(test_dataset))
print("Test loader ready ✅")

Test dataset size: 44
Test loader ready ✅


In [14]:
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,            # IMPORTANT: avoid multiprocessing
    collate_fn=r2gen_collate_fn
)

print("Test loader ready with custom collate_fn ✅")

Test loader ready with custom collate_fn ✅


In [15]:
sample = test_dataset[0]
print("Returned tuple lengths:", len(sample))
print("image_id:", sample[0])
print("image tensor shape:", sample[1].shape)  # should be [2,3,224,224]
print("seq_length:", sample[4])


Returned tuple lengths: 5
image_id: 1623
image tensor shape: torch.Size([2, 3, 224, 224])
seq_length: 21


7 — Load model + checkpoint

In [16]:
args.cmm_size = 2048
args.cmm_dim  = 512   # should match d_model

print("cmm_size:", args.cmm_size, "cmm_dim:", args.cmm_dim)

cmm_size: 2048 cmm_dim: 512


In [17]:
import torch

model = BaseCMNModel(args, tokenizer).to(DEVICE)

checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
state_dict = checkpoint["state_dict"] if isinstance(checkpoint, dict) and "state_dict" in checkpoint else checkpoint

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("✅ Model weights loaded")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

model.eval()
print("✅ Model set to eval()")

/home/psivakumar/.conda/envs/ml/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/psivakumar/.conda/envs/ml/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/psivakumar/.conda/envs/ml/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/tmp/ipykernel_525487/2488546187.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle mo

✅ Model weights loaded
Missing keys: 0
Unexpected keys: 0
✅ Model set to eval()


In [18]:
args.eos_idx = 1  # token id 1 corresponds to "."
print("eos_idx set to:", args.eos_idx)

eos_idx set to: 1


8 — Run inference on TEST and preview 3 outputs

In [19]:
results = []

with torch.no_grad():
    for batch in test_loader:
        image_ids, images, report_ids, report_masks, seq_lengths = batch
        images = images.to(DEVICE)

        # ---- FIX: unpack model output ----
        outputs = model(images, mode="sample")

        # outputs can be (seq, logprobs) or just seq
        if isinstance(outputs, tuple):
            output_ids = outputs[0]
        else:
            output_ids = outputs

        preds = tokenizer.decode_batch(output_ids.cpu().numpy())
        gts = tokenizer.decode_batch(report_ids[:, 1:].cpu().numpy())

        for uid, gt, pred in zip(image_ids, gts, preds):
            results.append({
                "uid": str(uid),
                "ground_truth": gt,
                "prediction": pred
            })

print("✅ Generated predictions:", len(results))

# Preview
for r in results[:3]:
    print("\nUID:", r["uid"])
    print("GT :", r["ground_truth"][:200])
    print("PR :", r["prediction"][:200])

✅ Generated predictions: 44

UID: 1623
GT : normal cardiomediastinal contours . no pneumothorax pleural effusions or focal lung consolidation . impression no acute cardiopulmonary abnormality .
PR : the heart and lungs have xxxx xxxx in the interval . both lungs are clear and expanded . heart and mediastinum normal . impression no active disease .

UID: 2730
GT : 2 images . heart size and pulmonary vascular engorgement appear within limits of normal . mediastinal contour is unremarkable . no focal consolidation pleural effusion or pneumothorax identified . no 
PR : the heart and lungs have xxxx xxxx in the interval . both lungs are clear and expanded . heart and mediastinum normal . impression no active disease .

UID: 1616
GT : chest . lungs are clear bilaterally . cardiac and mediastinal silhouettes are normal . pulmonary vasculature is normal . no pneumothorax or pleural effusion . no acute bony abnormality . ribs . there 
PR : the heart and lungs have xxxx xxxx in the interval . bo

In [20]:
import json
from pathlib import Path

OUT_DIR = Path("/home/psivakumar/thesis/code/R2GenCMN/records/iu_xray_r2gencmn_baseline_5ep")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "preds_test.json"

with open(OUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print("✅ Saved:", OUT_PATH)
print("Count:", len(results))

✅ Saved: /home/psivakumar/thesis/code/R2GenCMN/records/iu_xray_r2gencmn_baseline_5ep/preds_test.json
Count: 44


In [21]:
from pathlib import Path

rec_dir = Path("/home/psivakumar/thesis/code/R2GenCMN/records/iu_xray_r2gencmn_baseline_5ep")
print("Record dir exists:", rec_dir.exists())
print("Files:")
for p in rec_dir.glob("*"):
    print(" -", p.name)


Record dir exists: True
Files:
 - preds_test.json


In [22]:
pred_texts = [r["prediction"].strip() for r in results]
unique_preds = len(set(pred_texts))
print("Total preds:", len(pred_texts))
print("Unique preds:", unique_preds)

# show the 5 most common predictions
from collections import Counter
for text, cnt in Counter(pred_texts).most_common(5):
    print("\nCount:", cnt)
    print("Pred:", text[:200])


Total preds: 44
Unique preds: 14

Count: 21
Pred: the heart and lungs have xxxx xxxx in the interval . both lungs are clear and expanded . heart and mediastinum normal . impression no active disease .

Count: 6
Pred: the heart and lungs have xxxx xxxx xxxx in the interval . both lungs are clear and expanded . heart and mediastinum normal . impression no active disease .

Count: 2
Pred: the heart is normal in size . the mediastinum is stable . the aorta is atherosclerotic . there is no focal air space consolidation . there is no pleural effusion or pneumothorax . there is mild degene

Count: 2
Pred: the heart is normal in size . the mediastinum is stable . the aorta is atherosclerotic . the lungs are clear . impression no acute disease .

Count: 2
Pred: the heart is normal in size . the mediastinum is unremarkable . the lungs are clear . impression no acute disease .


In [23]:
# show first 10 UIDs from loader to confirm it changes
uids_seen = []
for i, batch in enumerate(test_loader):
    image_ids = batch[0]
    uids_seen.extend([str(x) for x in image_ids])
    if len(uids_seen) >= 20:
        break

print("First 20 UIDs from loader:", uids_seen[:20])
print("Unique in first 20:", len(set(uids_seen[:20])))


First 20 UIDs from loader: ['1623', '2730', '1616', '1199', '840', '2136', '87', '3109', '3530', '3320', '3523', '3848', '3975', '2462', '3796', '1040', '698', '3341', '403', '3283']
Unique in first 20: 20
